# Subgraph explorer (practice)

A sandbox for the expansion-test measurement (`scripts/data/subgraph.py`). It opens the duckdb file that
`scripts/data/measure.py` built under `NGQ_DATA_DIR/derived/` **read-only** (through views), so it writes
nothing to the repo or to the cached data. Change the seed, the definition or the pruning percentile and
see the candidate counts move; the last section swaps in a modified rule defined here.

Needs: `uv sync --group data`, the downloads (`scripts/data/download.py`) and one run of
`scripts/data/measure.py` for the graph you pick.

In [ ]:
import os, sys
from pathlib import Path

import duckdb

REPO = Path.cwd().parent if Path.cwd().name == "practice" else Path.cwd()
sys.path.insert(0, str(REPO / "scripts" / "data"))
import subgraph as sg
from subgraph import SubgraphSpec
from paths import derived_dir

print("NGQ_DATA_DIR =", os.environ.get("NGQ_DATA_DIR", "(unset: <repo>/data)"))

In [ ]:
GRAPH = "primekg"          # or "optimuskg"
MAIN = {"primekg": "28780", "optimuskg": "MONDO_0004975"}[GRAPH]

con = duckdb.connect()      # in-memory: scratch tables live here
con.execute(f"ATTACH '{derived_dir() / (GRAPH + '.duckdb')}' AS g (READ_ONLY)")
for t in ("nodes", "edges", "adj", "deg"):
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM g.{t}")
con.sql("SELECT count(*) AS nodes FROM nodes").show()

## 1. Seed candidates
Everything whose name mentions Alzheimer. Try adding or dropping nodes from `SEED` below.

In [ ]:
con.sql("""
SELECT n.id, n.name, n.type, d.degree FROM nodes n JOIN deg d USING (id)
WHERE n.name ILIKE '%alzheimer%' ORDER BY d.degree DESC
""").show(max_width=200)

In [ ]:
SEED = [MAIN]               # e.g. add subtype IDs from the table above
sg.set_seed(con, SEED)
con.sql("SELECT count(*) AS n1 FROM n1").show()

## 2. Counts per definition
The pruning cap is a full-graph degree percentile. Change `PCTS` and rerun.

In [ ]:
PCTS = [90, 95, 99]
specs = [SubgraphSpec("a", "a"), SubgraphSpec("b", "b")] + [
    SubgraphSpec(f"c{p}", "c", max_degree=sg.degree_threshold(con, p)) for p in PCTS
]
rows = []
for spec in specs:
    sg.build_subgraph(con, spec)
    facts = sg.single_hop_facts(con, spec.name)
    paths = sg.two_hop_paths(con, spec.name)
    un = sg.unanswerable_candidates(con, spec.name)
    n_paths = con.execute(f"SELECT count(*) FROM {paths}").fetchone()[0]
    top1 = con.execute(f"SELECT coalesce(max(c), 0) FROM (SELECT count(*) c FROM {paths} GROUP BY m)").fetchone()[0]
    rows.append({
        "def": spec.name, "cap": spec.max_degree,
        "edges": con.execute(f"SELECT count(*) FROM sub_{spec.name}").fetchone()[0],
        "facts": con.execute(f"SELECT count(*) FROM {facts}").fetchone()[0],
        "paths": n_paths, "top1_share": round(top1 / n_paths, 3) if n_paths else None,
        "unans_r_y": con.execute(f"SELECT count(*) FROM (SELECT DISTINCT r, y FROM {un})").fetchone()[0],
        "partial": sg.partial_candidates(con, spec.name),
    })
for r in rows:
    print(r)

## 3. Who are the hubs?
Top intermediates on two-hop paths for one definition.

In [ ]:
DEF = "b"
paths = sg.two_hop_paths(con, DEF)
con.sql(f"""
SELECT p.m, n.name, n.type, count(*) AS paths, count(DISTINCT p.t) AS endpoints
FROM {paths} p JOIN nodes n ON n.id = p.m GROUP BY 1, 2, 3 ORDER BY paths DESC LIMIT 15
""").show(max_width=200)

## 4. Swap in a modified rule

The project's unanswerable rule allows any subgraph node of a compatible type, however obscure. The
variant below drops candidates whose full-graph degree is under `MIN_DEG`, a rough proxy for entities
too obscure to phrase as a natural question. Change `MIN_DEG` or the rule and compare with the project's
count. (A "two-hop endpoints only" variant changes nothing for a single seed: in (b) and (c) every
non-seed node is either adjacent to the seed or a two-hop endpoint.)

In [ ]:
MIN_DEG = 20

def unanswerable_min_degree(con, name, min_deg):
    """Variant: like sg.unanswerable_candidates, but y needs full-graph degree >= min_deg."""
    t = sg.unanswerable_candidates(con, name)
    return con.execute(
        f"SELECT count(*) FROM (SELECT DISTINCT u.r, u.y FROM {t} u JOIN deg d ON d.id = u.y WHERE d.degree >= ?)",
        [min_deg],
    ).fetchone()[0]

for spec in specs:
    sg.build_subgraph(con, spec)
    print(spec.name, "project rule:", next(r["unans_r_y"] for r in rows if r["def"] == spec.name),
          f"| degree >= {MIN_DEG} variant:", unanswerable_min_degree(con, spec.name, MIN_DEG))